# Spatial Panel Models in spreg
This notebook presents an usage example of the spatial panel functions available in spreg as of version 1.9.0.
Prepared by: Luc Anselin and Pedro Amaral

# Preliminaries
Import required libraries and load the example data. 

In this example, we will use data on NCOVR US County Homicides (3085 areas) included in libpysal. We will extract the HR (homicide rates) data in the 70's, 80's and 90's from the dataframe and make them the dependent variable for the regression. Note that the data can also be passed in the long format instead of wide format (i.e. a vector with n\*t rows and a single column for the dependent variable and a matrix of dimension n\*t x k for the independent variables).

For explanatory variables, we will extract RD and PS in the same time periods from the dataframe to be used as independent variables in the regression.

IMPORTANT: If in wide format as in the example below, the data must be organized in a way that all time periods of a given variable are side-by-side and in the correct time order. That is: x[:, 0:T] refers to T periods of k1, x[:, T+1:2T] refers to k2, etc. By default a vector of ones will be added to the independent variables passed in.


In [2]:
import libpysal
import spreg
import geopandas as gpd

/opt/miniconda3/envs/env14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
libpysal.examples.load_example('NCOVR')
shp_path = libpysal.examples.get_path('NAT.shp')
dfs = gpd.read_file(shp_path)

# Subset to East Coast states to have a smaller sample
east_fips = [9, 10, 11, 12, 13, 23, 24, 25, 33, 34, 36, 37, 42, 44, 45, 50, 51, 54]
dfs2 = dfs[dfs['STFIPS'].isin(east_fips)]

wq1 = libpysal.weights.Queen.from_dataframe(dfs2,ids='FIPSNO')
wq1.transform = 'r'

y_var = dfs2[['HR70', 'HR80', 'HR90']]
x_var = dfs2[['RD70','RD80','RD90','PS70','PS80','PS90']]

spreg version 1.9.0 offers the following panel functions (in addition to the SUR estimators):

1. OLS and Basic Panel Classes
- PooledOLS
    Description: Baseline pooled regression for panel data. Initializes and runs the pooled OLS estimation with optional BSK spatial diagnostics.

- PanelFE
    Description: Fixed Effects (Within) estimator for panel data. Performs the "Within Transformation" (demeaning) and executes the regression.

- PanelRE
    Description: Random Effects (GLS) estimator. Estimates variance components using Swamy-Arora, computes quasi-demeaning, and performs the Hausman test.


2. GMM-Based Spatial Error Classes
- GM_ErrorPooled
    Description: Pooled Spatial Error Model (SEM) via GMM. Based on the heteroskedasticity-robust estimator proposed by `Arraiz2010`, available in the spreg function `GM_Error_Het`.

- GM_ErrorRE
    Description: KKP spatial random effects model.


3. Maximum Likelihood (ML) Spatial Classes
- ML_ErrorPooled
    Description: Pooled ML SEM with diagnostic output.

- ML_ErrorFE
    Description: ML estimation for Fixed Effects Spatial Error models.

- ML_ErrorRE
    Description: ML estimation for Random Effects Spatial Error models using Ord's eigenvalue approach.

- ML_LagFE
    Description: ML estimation for Fixed Effects Spatial Lag models with spatial impacts.

- ML_LagRE
    Description: ML Spatial Lag Random Effects to estimate both the spatial lag (ρ) and the random effects component (ϕ)

In the examples shown below, the following notation will be used:
* $Y$: $NT \times 1$ vector of the dependent variable
* $X$: $NT \times K$ matrix of exogenous variables
* $\beta$: $K \times 1$ vector of parameters
* $W_N$: $N \times N$ spatial weights matrix
* $\iota_T$: $T \times 1$ vector of ones
* $\otimes$: Kronecker product
* $\mu$: $N \times 1$ vector of individual fixed or random effects
* $\epsilon$: $NT \times 1$ vector of idiosyncratic errors
* $\rho$: Spatial lag parameter
* $\lambda$: Spatial error parameter

# Pooled panel

$$Y = X\beta + \epsilon$$
*where $\epsilon \sim N(0, \sigma^2 I_{NT})$*

Optional arguments include (among others):
- spat_diag: If True, then compute the BSK tests (boolean, default = True)
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- BSK_list: List of BSK tests to compute if spat_diag is True. 
                     Options are "all", "LMJ", "LM1", "LM2", "LMC_spatial", "LMC_RE" 
                     (default: ["LMJ", "LM1", "LM2"]).


In [4]:
panel = spreg.PooledOLS(y=y_var, x=x_var, w=wq1, spat_diag=True, slx_lags=0)
print(panel.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: POOLED OLS PANEL
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           3
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2364
R-squared           :      0.3641
Adjusted R-squared  :      0.3636
Sum squared residual:     83215.9                F-statistic           :    676.8331
Sigma-square        :      35.201                Prob(F-statistic)     :  3.906e-233
S.E. of regression  :       5.933                Log likelihood        :   -7571.669
Sigma-square ML     :      35.157                Akaike info criterion :   15149.338
S.E of regression ML:      5.9293                Schwarz criterion     :   15166.646

-----------------------

# Fixed Effects

$$Y = (\iota_T \otimes I_N)\mu + X\beta + \epsilon$$

*where $\iota_T$ is a $T \times 1$ vector of ones, and $\mu$ is an $N \times 1$ vector of fixed effects.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 

In [5]:
panelFE = spreg.PanelFE(y=y_var, x=x_var, w=wq1)
print(panelFE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: FIXED EFFECTS PANEL (ONE-WAY)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           2
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2365
Pseudo R-squared    :      0.0362
Sum squared residual:      612992                F-statistic           :   2056.4811
Sigma-square        :      23.750                Prob(F-statistic)     :           0
S.E. of regression  :       4.873                Log likelihood        :   -9935.016
Sigma-square ML     :     258.974                Akaike info criterion :   19874.032
S.E of regression ML:     16.0927                Schwarz criterion     :   19885.571
Fixed-effects mean  :     12.9384

----------

In [6]:
# Individual effects (mu_i) for the first 5 observations
panelFE.mu_i[0:5]

array([[ 3.36232066],
       [ 1.60766964],
       [-0.51299016],
       [ 8.92060208],
       [ 2.99745542]])

# Random Effects

$$Y = X\beta + v$$
$$v = (\iota_T \otimes I_N)\mu + \epsilon$$

*where $\mu \sim N(0, \sigma_\mu^2 I_N)$ represents the random effects, and $\epsilon \sim N(0, \sigma_\epsilon^2 I_{NT})$.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- spat_diag: If True (default), then compute the 'LMC_spatial' BSK test (boolean).

In [7]:
panelRE = spreg.PanelRE(y=y_var, x=x_var, w=wq1)
print(panelRE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: RANDOM EFFECTS (ONE-WAY) PANEL
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           3
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2364
Pseudo R-squared    :      0.3639
Sum squared residual:     83401.1                F-statistic           :    600.5147
Sigma-square        :      24.807                Prob(F-statistic)     :  1.296e-211
S.E. of regression  :       4.981                Log likelihood        :   -7574.300
Sigma-square ML     :      24.807                Akaike info criterion :   15154.600
S.E of regression ML:      4.9806                Schwarz criterion     :   15171.908

-------------------------------------------

# Pooled SEM - GM estimator

$$Y = X\beta + u$$
$$u = \lambda(I_T \otimes W_N)u + \epsilon$$

*where $\lambda$ is the spatial error coefficient and $W_N$ is the spatial weights matrix.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- max_iter: Maximum number of iterations of steps 2a and 2b from `Arraiz2010` (int, default = 1).
- step1c: If True, then include Step 1c from `Arraiz2010` (boolean, default = False).
- nonspat_diag: If True (default), then compute the 'LMC_RE' BSK test.

In [8]:
panelSEM = spreg.GM_ErrorPooled(y=y_var, x=x_var, w=wq1)
print(panelSEM.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GMM POOLED SPATIAL ERROR MODEL (SEM)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           3
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2364
Pseudo R-squared    :      0.3636
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         7.54099         0.27761        27.16348         0.00000
                  RD         4.52263 

# Pooled SEM - ML estimator

$$Y = X\beta + u$$
$$u = \lambda(I_T \otimes W_N)u + \epsilon$$

*where $\lambda$ is the spatial error coefficient and $W_N$ is the spatial weights matrix.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- method: if 'LU' (default), LU decomposition for sparse matrices (Recommended for Panel); 
                   if 'ord', Ord eigenvalue calculation (Slow for large Panel); 
                   if 'full', brute force (Should not use for Panel).

- nonspat_diag: If True (default), then compute the 'LMC_RE' BSK test.                   

In [9]:
panelML_SEM = spreg.ML_ErrorPooled(y=y_var, x=x_var, w=wq1)
print(panelML_SEM.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML POOLED SPATIAL ERROR MODEL (SEM)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :     dep_var                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           3
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2364
Pseudo R-squared    :      0.3636
Log likelihood      :  -7416.9763
Sigma-square ML     :     29.5004                Akaike info criterion :   14839.953
S.E of regression   :      5.4314                Schwarz criterion     :   14857.261

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
    

# Error model - Random Effects - KKP GM Estimator

$$Y = X\beta + u$$
$$u = \lambda(I_T \otimes W_N)u + v$$
$$v = (\iota_T \otimes I_N)\mu + \epsilon$$

*where the spatial error process ($\lambda$) applies to the composite error term, meaning both the random effects ($\mu$) and idiosyncratic errors ($\epsilon$) are spatially correlated.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- full_weights: (boolean) Considers different weights for each of the 6 moment
                  conditions if True or only 2 sets of weights for the
                  first 3 and the last 3 moment conditions if False (default)

In [10]:
panelSEM_RE = spreg.GM_ErrorRE(y=y_var, x=x_var, w=wq1)
print(panelSEM_RE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GMM SPATIAL ERROR PANEL MODEL - RANDOM EFFECTS (KKP)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           3
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2364
Pseudo R-squared    :      0.3634

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         7.61779         0.26073        29.21762         0.00000
                  RD         4.33248         0.17641        24.55953         0.00000
                  PS 

# Error model - Random Effects - ML Estimator

$$Y = X\beta + v$$
$$v = (\iota_T \otimes I_N)\mu + u$$
$$u = \lambda(I_T \otimes W_N)u + \epsilon$$

*where the spatial error process ($\lambda$) applies only to the idiosyncratic error term ($u$). The individual random effects ($\mu$) remain spatially independent.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 

In [11]:
panelSEM_RE = spreg.ML_ErrorRE(y=y_var, x=x_var, w=wq1)
print(panelSEM_RE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR PANEL MODEL (SEM) - RANDOM EFFECTS
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      8.8420                Number of Variables   :           3
S.D. dependent var  :      7.4372                Degrees of Freedom    :        2364
Pseudo R-squared    :      0.3638

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         7.58728         0.22993        32.99879         0.00000
                  RD         4.39257         0.17090        25.70273         0.00000
                  PS  

# Error Model - Fixed Effects - ML Estimator

$$Y = (\iota_T \otimes I_N)\mu + X\beta + u$$
$$u = \lambda(I_T \otimes W_N)u + \epsilon$$

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 

In [12]:
panelSEM_FE = spreg.ML_ErrorFE(y=y_var, x=x_var, w=wq1)
print(panelSEM_FE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR PANEL MODEL - FIXED EFFECTS
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      0.0000                Number of Variables   :           2
S.D. dependent var  :      4.0669                Degrees of Freedom    :        2365
Pseudo R-squared    :      0.0422
Log likelihood      :  -6608.0839
Sigma-square ML     :     15.4572                Akaike info criterion :   13220.168
S.E of regression   :      3.9316                Schwarz criterion     :   13231.706
Fixed-effects mean  :     12.7425

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
----------------------------------------------

In [13]:
# Individual effects (mu_i) for the first 5 observations
panelSEM_FE.mu_i[0:5]

array([[ 3.31195411],
       [ 1.67335124],
       [-0.18020496],
       [ 8.72536659],
       [ 3.03582746]])

# Lag Model - Random Effects - ML Estimator

$$Y = \rho(I_T \otimes W_N)Y + X\beta + v$$
$$v = (\iota_T \otimes I_N)\mu + \epsilon$$

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- spat_impacts: Include average direct impact (ADI), average indirect impact (AII),
                    and average total impact (ATI) in summary results;
                    Options are 'simple', 'full', 'power', 'all' or None;
                    (str or list, default = 'simple').


In [14]:
panelSAR_RE = spreg.ML_LagRE(y=y_var, x=x_var, w=wq1)
print(panelSAR_RE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL LAG PANEL - RANDOM EFFECTS
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      6.5609                Number of Variables   :           4
S.D. dependent var  :      6.1552                Degrees of Freedom    :        2363
Pseudo R-squared    :      0.3692
Log likelihood      :  -7383.2780
Sigma-square ML     :     23.9269                Akaike info criterion :   14774.556
S.E of regression   :      4.8915                Schwarz criterion     :   14797.633

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
  

# Lag Model - Fixed Effects - ML Estimator

$$Y = \rho(I_T \otimes W_N)Y + (\iota_T \otimes I_N)\mu + X\beta + \epsilon$$

*where $\rho$ is the spatial lag (autoregressive) coefficient.*

Optional arguments include (among others):
- slx_lags: Number of spatial lags of X to include in the model specification. If slx_lags>0, the specification becomes of the SLX type (integer, default = 0).
- slx_vars: "all" or list of booleans to select x variables to be lagged (str or list, default = "all"). 
- spat_impacts: Include average direct impact (ADI), average indirect impact (AII),
                    and average total impact (ATI) in summary results;
                    Options are 'simple', 'full', 'power', 'all' or None;
                    (str or list, default = 'simple').


In [15]:
panelSAR_FE = spreg.ML_LagFE(y=y_var, x=x_var, w=wq1)
print(panelSAR_FE.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL LAG PANEL - FIXED EFFECTS
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :          HR                Number of Observations:        2367
Mean dependent var  :      0.0000                Number of Variables   :           3
S.D. dependent var  :      4.0669                Degrees of Freedom    :        2364
Pseudo R-squared    :      0.0659
Log likelihood      :  -6607.9807
Sigma-square ML     :     15.4592                Akaike info criterion :   13221.961
S.E of regression   :      3.9318                Schwarz criterion     :   13239.269
Fixed-effects mean  :     12.4434

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------

In [16]:
# Individual effects (mu_i) for the first 5 observations
panelSAR_FE.mu_i[0:5]

array([[3.16956916],
       [1.72726064],
       [0.25762143],
       [8.24781692],
       [3.18171931]])